# α-DPI for EHT Black Hole Feature Extraction
### Geometric Parameter Inference via α-Divergence Variational Inference

---

This notebook demonstrates posterior inference over geometric black hole parameters
from EHT closure quantities using the α-DPI method (Sun et al. 2022, ApJ 932:99).

Unlike pixel-space imaging (DPI), α-DPI fits a **parametric geometric model** directly
to closure data. The output is a posterior distribution over interpretable physical
features such as ring diameter, width, brightness asymmetry, and orientation.

**Contents**
1. [Setup & Data Loading](#1.-Setup-&-Data-Loading)
2. [Geometric Model & Parameter Space](#2.-Geometric-Model-&-Parameter-Space)
3. [Load Pretrained Model](#3.-Load-Pretrained-Model)
4. [Posterior Sampling & Importance Reweighting](#4.-Posterior-Sampling-&-Importance-Reweighting)
5. [Corner Plot of Posterior Parameters](#5.-Corner-Plot)
6. [Posterior Image Samples](#6.-Posterior-Image-Samples)
7. [Parameter Recovery Metrics](#7.-Parameter-Recovery-Metrics)
8. [ELBO Model Selection](#8.-ELBO-Model-Selection)
9. [Training from Scratch (Optional)](#9.-Training-from-Scratch)

In [1]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

TASK_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if TASK_ROOT not in sys.path:
    sys.path.insert(0, TASK_ROOT)

from src.preprocessing import prepare_data, load_ground_truth, load_metadata
from src.solvers import RealNVP, AlphaDPISolver
from src.physics_model import SimpleCrescentNuisanceFloorParam2Img, NUFFTForwardModel
from src.visualization import (
    plot_corner, plot_elbo_comparison, plot_posterior_images,
    plot_loss_curves, compute_feature_metrics, print_feature_metrics,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REF_DIR = os.path.join(TASK_ROOT, 'evaluation', 'reference_outputs')
print(f'Setup complete. Device: {device}')

Welcome to eht-imaging!

Setup complete. Device: cuda


---
## 1. Setup & Data Loading

Load the EHT observation (UVFITS), extract closure quantities (closure phases
and log closure amplitudes), and compute NUFFT parameters for the forward model.

Closure quantities are **gain-invariant** observables formed from triplets (closure phase)
and quadruplets (closure amplitude) of baselines, making them robust to station-based
calibration errors.

In [ ]:
(obs, obs_data, closure_indices, nufft_params,
 flux_const, metadata) = prepare_data(
    os.path.join(TASK_ROOT, 'data'))

npix = metadata['npix']
fov_uas = metadata['fov_uas']
pixel_size_uas = fov_uas / npix
extent_uas = fov_uas / 2

print(f'Image size       : {npix} x {npix}')
print(f'FOV              : {fov_uas} \u03bcas')
print(f'Pixel size       : {pixel_size_uas:.2f} \u03bcas/pixel')
print(f'Visibilities     : {len(obs_data["vis"])}')
print(f'Closure phases   : {len(closure_indices["cphase_data"]["cphase"])}')
print(f'Closure amps     : {len(closure_indices["camp_data"]["camp"])}')
print(f'Estimated flux   : {flux_const:.4f} Jy')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Ground truth image
gt_image = load_ground_truth(os.path.join(TASK_ROOT, 'data'), npix, fov_uas)
extent = [extent_uas, -extent_uas, -extent_uas, extent_uas]
axes[0].imshow(gt_image, origin='lower', cmap='afmhot', extent=extent)
axes[0].set_xlabel('RA (\u03bcas)')
axes[0].set_ylabel('Dec (\u03bcas)')
axes[0].set_title('Ground Truth Image')

# uv-coverage
uv = obs_data['uv_coords'] / 1e9
axes[1].scatter(uv[:, 0], uv[:, 1], s=0.5, alpha=0.5, c='steelblue')
axes[1].scatter(-uv[:, 0], -uv[:, 1], s=0.5, alpha=0.5, c='coral')
axes[1].set_xlabel('u (G\u03bb)')
axes[1].set_ylabel('v (G\u03bb)')
axes[1].set_title('uv-Coverage')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Geometric Model & Parameter Space

The geometric model parameterizes the black hole image as a **crescent** (asymmetric ring)
blended with a **tapered floor disk**, plus $N$ **rotated elliptical Gaussian** nuisance
components. This follows Figure 6 of Sun et al. (2022).

### Crescent + floor parameters (4 + 2 = 6 dimensions)

The crescent is a Gaussian ring modulated by an asymmetry factor $(1 + A\cos(\theta - \varphi))$,
blended with a tapered disk via a floor fraction:

$$I_{\text{cres}} = f_{\text{cres}} \left[(1 - f_{\text{floor}}) \cdot \hat{I}_{\text{ring}} + f_{\text{floor}} \cdot \hat{I}_{\text{disk}}\right]$$

| Index | Parameter | Symbol | Range | Description |
|-------|-----------|--------|-------|-------------|
| 0 | **Diameter** | $d$ | [20, 80] μas | Outer ring diameter $d = 2r$, where $r \in$ [10, 40] μas |
| 1 | **Width** | $w$ | [1, 40] μas | Ring thickness (Gaussian $\sigma$ of radial profile) |
| 2 | **Asymmetry** | $A$ | [0.001, 0.99] | Brightness contrast; 0 = uniform ring, 1 = one-sided crescent |
| 3 | **Position angle** | $\varphi$ | [-181, 181] deg | Orientation of the bright side, measured E of N |
| 16 | **Floor** | $f_{\text{floor}}$ | [0, 1] | Fraction of flux in the tapered disk component |
| 17 | **Crescent flux** | $f_{\text{cres}}$ | [0.001, 2.0] | Overall flux scale of the crescent+disk |

### Gaussian nuisance parameters (6 per component)

Each of the $N$ rotated elliptical Gaussians absorbs compact emission not captured by the crescent.
Unlike covariance-based parameterization, these use an explicit rotation angle $\theta$:

| Index offset | Parameter | Range | Description |
|-------------|-----------|-------|-------------|
| +0 | $x$ | [-200, 200] μas | RA offset from image center |
| +1 | $y$ | [-200, 200] μas | Dec offset from image center |
| +2 | $f$ | [0.001, 2.0] | Flux scale of this component |
| +3 | $\sigma_x$ | [1, 100] μas | Semi-major axis width |
| +4 | $\sigma_y$ | [1, 100] μas | Semi-minor axis width |
| +5 | $\theta$ | [0, 90.5] deg | Rotation angle of the ellipse |

For the default configuration with $N=2$ Gaussians, the total parameter space is
$4 + 6 \times 2 + 2 = 18$ dimensions (indices 0–3: crescent, 4–15: Gaussians, 16–17: floor+flux).

### Flow architecture

A Real-NVP normalizing flow maps an 18-D Gaussian latent $\mathbf{z}$ to unconstrained
parameters $\mathbf{u}$, which are then passed through a sigmoid $\sigma(\mathbf{u}) \in (0,1)^{18}$
and linearly rescaled to physical ranges. The geometric model renders these parameters
into a 64×64 image, which is then Fourier-transformed via NUFFT to predict closure quantities.

In [ ]:
gt_params_dict = metadata.get('ground_truth_params', {})

print('Ground truth crescent parameters:')
print(f'  Diameter  : {gt_params_dict.get("diameter_uas", 44.0):.1f} \u03bcas')
print(f'  Width     : {gt_params_dict.get("width_uas", 11.36):.2f} \u03bcas')
print(f'  Asymmetry : {gt_params_dict.get("asymmetry", 0.5):.2f}')
print(f'  PA        : {gt_params_dict.get("position_angle_deg", -90.5):.1f} deg')
print(f'  N_gaussian: {gt_params_dict.get("n_gaussian", 2)}')

---
## 3. Load Pretrained Model

Load a pretrained α-DPI model (Real-NVP normalizing flow) that maps an
18-D Gaussian latent space to geometric crescent + floor + Gaussian parameters.

In [ ]:
n_gaussian = metadata.get('n_gaussian', 2)

solver = AlphaDPISolver(
    npix=npix, fov_uas=fov_uas,
    n_flow=metadata['n_flow'],
    seqfrac=1.0 / metadata.get('seqfrac_inv', 16),
    geometric_model=metadata.get('geometric_model', 'simple_crescent_floor_nuisance'),
    n_gaussian=n_gaussian,
    r_range=metadata.get('r_range', [10.0, 40.0]),
    width_range=metadata.get('width_range', [1.0, 40.0]),
    device=device,
)

# Build the geometric model and flow
nparams = solver._build_geometric_model()
solver.params_generator = RealNVP(
    nparams, solver.n_flow, affine=True,
    seqfrac=solver.seqfrac, permute='random', batch_norm=True
).to(solver.device)

# Load pretrained weights
model_path = os.path.join(REF_DIR, 'params_generator_state_dict.pt')
solver.params_generator.load_state_dict(
    torch.load(model_path, map_location=device, weights_only=True))
solver.params_generator.eval()

print(f'Geometric model  : crescent + floor + {n_gaussian} Gaussians')
print(f'Parameter space  : {nparams} dimensions (4 crescent + {6*n_gaussian} Gaussian + 2 floor/flux)')
print(f'Flow blocks      : {metadata["n_flow"]}')
print(f'Model parameters : {sum(p.numel() for p in solver.params_generator.parameters()):,}')

---
## 4. Posterior Sampling & Importance Reweighting

α-DPI is a **two-step** algorithm:
1. **Step 1 (Training)**: Minimize α-divergence between the flow $q_\theta(\mathbf{x})$ and the true posterior
2. **Step 2 (Importance sampling)**: Draw $N$ samples from the trained $q_\theta$, then reweight each
   sample by the importance ratio $w_i \propto p(\mathbf{y}|\mathbf{x}_i) \, p(\mathbf{x}_i) / q_\theta(\mathbf{x}_i)$

The importance weights correct any remaining mismatch between $q_\theta$ and the true posterior.
The **effective sample size** $N_{\text{eff}} = 1 / \sum w_i^2$ indicates how many independent
samples the reweighted distribution is equivalent to.

In [ ]:
posterior = solver.importance_resample(
    obs_data, closure_indices, nufft_params, n_samples=10000
)

params_physical = posterior['params_physical']
importance_weights = posterior['importance_weights']
posterior_images = posterior['images']
weighted_mean_image = posterior['weighted_mean_image']

n_eff = 1.0 / np.sum(importance_weights**2)
print(f'Posterior samples : {params_physical.shape[0]}')
print(f'Parameter dims   : {params_physical.shape[1]} (4 crescent + {6*n_gaussian} Gaussian + 2 floor/flux)')
print(f'Effective samples: {n_eff:.0f} / {len(importance_weights)}')
print(f'Max weight       : {importance_weights.max():.4f}')

---
## 5. Corner Plot

Posterior distributions of the 4 crescent parameters. Red dashed lines indicate
the ground truth values. Off-diagonal panels show 2D scatter plots of sample pairs;
diagonal panels show 1D marginal histograms weighted by importance weights.

This corresponds to **Figure 7** in Sun et al. (2022).

In [ ]:
gt_array = np.array([
    gt_params_dict.get('diameter_uas', 44.0),
    gt_params_dict.get('width_uas', 11.36),
    gt_params_dict.get('asymmetry', 0.5),
    gt_params_dict.get('position_angle_deg', -90.5),
])
crescent_param_names = ['diameter (\u03bcas)', 'width (\u03bcas)', 'asymmetry', 'PA (deg)']

# Only plot the first 4 columns (crescent params) for readability
fig = plot_corner(
    params_physical[:, :4],
    param_names=crescent_param_names,
    ground_truth=gt_array,
    importance_weights=importance_weights,
)
fig.suptitle('\u03b1-DPI Posterior: Crescent Parameters (Figure 7)', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Print weighted posterior summary vs ground truth
w = importance_weights / importance_weights.sum()
print(f'{"Parameter":<20s} {"Ground Truth":>14s} {"Posterior Mean":>14s} {"Posterior Std":>14s}')
print('-' * 64)
for i, name in enumerate(crescent_param_names):
    gt_val = gt_array[i]
    post_mean = np.sum(w * params_physical[:, i])
    post_std = np.sqrt(np.sum(w * (params_physical[:, i] - post_mean)**2))
    print(f'{name:<20s} {gt_val:>14.2f} {post_mean:>14.2f} {post_std:>14.2f}')

---
## 6. Posterior Image Samples

Images generated by passing posterior parameter samples through the geometric
crescent model. Each sample produces a different crescent with slightly different
diameter, width, asymmetry, and orientation. The **weighted mean image** aggregates
all importance-weighted samples.

In [ ]:
fig = plot_posterior_images(
    posterior_images, n_show=8,
    pixel_size_uas=pixel_size_uas,
    importance_weights=importance_weights,
)
fig.suptitle('Posterior Image Samples (importance-weighted)', y=1.02, fontsize=13)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

extent = [extent_uas, -extent_uas, -extent_uas, extent_uas]

im0 = axes[0].imshow(gt_image, origin='lower', cmap='afmhot', extent=extent)
axes[0].set_title('Ground Truth')
axes[0].set_xlabel('RA (\u03bcas)')
axes[0].set_ylabel('Dec (\u03bcas)')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(weighted_mean_image, origin='lower', cmap='afmhot', extent=extent)
axes[1].set_title('Weighted Mean (importance sampling)')
axes[1].set_xlabel('RA (\u03bcas)')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

residual = gt_image / gt_image.sum() - weighted_mean_image / weighted_mean_image.sum()
vlim = np.abs(residual).max()
im2 = axes[2].imshow(residual, origin='lower', cmap='RdBu_r',
                      extent=extent, vmin=-vlim, vmax=vlim)
axes[2].set_title('Residual (normalized)')
axes[2].set_xlabel('RA (\u03bcas)')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

---
## 7. Parameter Recovery Metrics

Evaluate how well the posterior recovers the ground truth geometric parameters:

- **Bias**: weighted posterior mean minus ground truth
- **Std**: weighted posterior standard deviation (uncertainty)
- **1σ Coverage**: fraction of samples within 1 standard deviation of the ground truth
  (ideally ~0.68 for Gaussian posteriors)

In [ ]:
metrics = compute_feature_metrics(
    params_physical, gt_array,
    importance_weights=importance_weights,
    param_names=crescent_param_names,
)

print('Parameter Recovery Metrics (crescent parameters):')
print('=' * 56)
print_feature_metrics(metrics)

---
## 8. ELBO Model Selection

The ELBO (Evidence Lower Bound) is used for **model selection**: compare crescent
models with different numbers of Gaussian components ($N = 0, 1, 2, 3$).
The model with the highest ELBO best balances data fit and model complexity.

$$\text{ELBO} = \mathbb{E}_{q_\theta}\big[\log p(\mathbf{y}|\mathbf{x})\big] - D_{\text{KL}}(q_\theta \| p)$$

In [ ]:
# Compute ELBO for the currently loaded model
elbo = solver.compute_elbo(
    obs_data, closure_indices, nufft_params, n_samples=10000
)
print(f'ELBO (n_gaussian={n_gaussian}): {elbo:.4f}')
print()
print('To perform full model selection, train separate models with')
print('n_gaussian = 0, 1, 2, 3 and compare their ELBOs.')
print('The model with highest ELBO is preferred (see Section 9).')

# Plot ELBO for the trained model
elbos = {f'{n_gaussian} Gauss (trained)': elbo}
fig = plot_elbo_comparison(elbos)
plt.show()

---
## 9. Training from Scratch (Optional)

To train α-DPI from scratch instead of using the pretrained checkpoint,
uncomment and run the cell below.

Key hyperparameters:
- `n_epoch=10000`: training iterations
- `batch_size=2048`: samples per gradient step
- `alpha=1.0, beta=1.0`: α-divergence parameter (effective α = 1 − β/N_data)
- `start_order=4, decay_rate=2000`: data warmup schedule $w(k) = \min(10^{-4+k/2000}, 1)$

In [ ]:
# # Uncomment to train from scratch (~30-60 min on GPU):
# solver_train = AlphaDPISolver(
#     npix=npix, fov_uas=fov_uas,
#     n_flow=metadata['n_flow'],
#     seqfrac=1.0 / metadata.get('seqfrac_inv', 16),
#     n_epoch=metadata['n_epoch'],
#     batch_size=metadata['batch_size'],
#     lr=metadata['lr'],
#     logdet_weight=metadata['logdet_weight'],
#     grad_clip=metadata['grad_clip'],
#     alpha=metadata.get('alpha_divergence', 1.0),
#     beta=metadata.get('beta', 1.0),
#     start_order=metadata.get('start_order', 4),
#     decay_rate=metadata.get('decay_rate', 2000),
#     geometric_model='simple_crescent_floor_nuisance',
#     n_gaussian=n_gaussian,
#     device=device,
# )
# result = solver_train.reconstruct(
#     obs_data, closure_indices, nufft_params, flux_const)
#
# plot_loss_curves(result['loss_history'])
# plt.show()

### Loss Curves (from reference training)

The training loss has three components:
- **Closure phase loss**: $\chi^2$ of predicted vs observed closure phases
- **Log closure amplitude loss**: $\chi^2$ of predicted vs observed log closure amplitudes
- **Entropy term** ($-\log\det$): encourages diversity in the posterior

In [ ]:
loss_path = os.path.join(REF_DIR, 'loss_history.npy')
if os.path.exists(loss_path):
    loss_history = np.load(loss_path, allow_pickle=True).item()
    fig = plot_loss_curves(loss_history)
    plt.show()

    print(f'Total epochs     : {len(loss_history["total"])}')
    n_tail = min(100, len(loss_history['total']))
    print(f'Final total loss : {np.mean(loss_history["total"][-n_tail:]):.4f}')
    print(f'Final cphase loss: {np.mean(loss_history["cphase"][-n_tail:]):.4f}')
    print(f'Final logca loss : {np.mean(loss_history["logca"][-n_tail:]):.4f}')
else:
    print('Loss history not available (run training first).')

---
## Summary

| Aspect | Detail |
|--------|--------|
| **Method** | α-DPI: α-divergence variational inference + importance sampling |
| **Data** | Closure phases + log closure amplitudes (gain-invariant) |
| **Geometric model** | Crescent + floor disk (6 params) + $N$ rotated elliptical Gaussians (6 params each) |
| **Architecture** | Real-NVP flow (16 blocks) → sigmoid → geometric model → NUFFT → closure quantities |
| **Parameter space** | 18-D for $N=2$: 4 crescent + 12 Gaussian + 2 floor/flux |
| **Training** | α-divergence with data-dependent warmup $w(k) = \min(10^{-4+k/2000}, 1)$ |
| **Post-processing** | Importance sampling reweighting for accurate posteriors |
| **Model selection** | ELBO comparison across $N = 0, 1, 2, 3$ Gaussian components |
| **Output** | Full posterior over geometric features with calibrated uncertainties |

**Reference:** Sun, H., Bouman, K.L., Tiede, P. et al. (2022).
*α-Deep Probabilistic Inference (α-DPI): Efficient Uncertainty Quantification from Exoplanet Astrometry to Black Hole Feature Extraction.*
ApJ 932:99. [arXiv:2201.08506](https://arxiv.org/abs/2201.08506)